In [2]:
import os
import shutil

import kagglehub
import pandas as pd

if not os.path.isdir("nlp-getting-started"):
    path = kagglehub.competition_download("nlp-getting-started")
    shutil.move(path, "nlp-getting-started")

df_train = pd.read_csv("nlp-getting-started/train.csv")
df_test = pd.read_csv("nlp-getting-started/test.csv")

df_train

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [93]:
import re
import gzip
import json
import time
import uuid

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from geopy.geocoders import Nominatim
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def tokenize(doc):
     return re.findall(r"\w+", doc) # TODO: make better tokenizer

class D2VVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, **kwargs):
        self.model = None
        self.hyperparameters = kwargs
    
    def fit(self, X, y=None):
        print(X, type(X))
        self.model = Doc2Vec([TaggedDocument(tokenize(x), [i]) for i, x in enumerate(X)], **self.hyperparameters)
        return self

    def transform(self, X, y=None):
        print(X, type(X))
        return np.vstack([self.model.infer_vector(tokenize(x)) for x in X])

class GeopyLocator(BaseEstimator, TransformerMixin):
    def __init__(self, **kwargs):
        try:
            with gzip.open("geopy_cache.json.gz", "rt") as file:
                self.cache = json.load(file)
        except FileNotFoundError:
            self.cache = {}
            
        self.geolocator = Nominatim(user_agent="ece_6420_final")
    
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        locations = []
        dirty = False
        
        for x in X:
            if x != x: # NaN
                locations.append([np.nan, np.nan])
            elif x in self.cache:
                if self.cache[x] is None:
                    locations.append([np.nan, np.nan])
                else:
                    locations.append(self.cache[x])
            else:
                print("Geocoding ", x, "...")
                location = self.geolocator.geocode(x)
                dirty = True
                time.sleep(1) # Respect Nominatim usage policy

                if location is None:
                    self.cache[x] = None
                    locations.append([np.nan, np.nan])
                else:
                    self.cache[x] = [location.latitude, location.longitude]
                    locations.append([location.latitude, location.longitude])
        
        if dirty:
            with gzip.open("geopy_cache.json.gz", "wt") as file:
                json.dump(self.cache, file)
        
        y = np.array(locations)
        return y


    
ct = make_column_transformer(
    ("drop", ["id"]),
    (OneHotEncoder(drop=[np.nan]), ["keyword"]),
    (make_pipeline(GeopyLocator(), SimpleImputer(), StandardScaler()), "location"),
    (D2VVectorizer(), "text")
)

ct.fit_transform(df_train.iloc[:100])

Geocoding  Live On Webcam ...
Geocoding  milky way ...
Geocoding  GREENSBORO,NORTH CAROLINA ...
Geocoding  England. ...
Geocoding  Sheffield Township, Ohio ...
Geocoding  India ...
Geocoding  Barbados ...
Geocoding  Anaheim ...
Geocoding  Abuja ...
Geocoding  USA ...
Geocoding  South Africa ...
Geocoding  Sao Paulo, Brazil ...
Geocoding  hollywoodland  ...
Geocoding  Edmonton, Alberta - Treaty 6 ...
Geocoding  Inang Pamantasan ...
Geocoding  Twitter Lockout in progress ...
Geocoding  Concord, CA ...
Geocoding  Calgary, AB ...
Geocoding  San Francisco ...
Geocoding  CLVLND ...
Geocoding  Nashville, TN ...
Geocoding  Santa Clara, CA ...
Geocoding  UK ...
Geocoding  St. Louis, MO ...
Geocoding  Walker County, Alabama ...
Geocoding  Australia ...
Geocoding  North Carolina ...
Geocoding  Norf Carolina ...
Geocoding  San Mateo County, CA ...
Geocoding  Njoro, Kenya ...
Geocoding  Your Sister's Bedroom ...
Geocoding  Arlington, TX ...
Geocoding  South Bloomfield, OH ...
Geocoding  New Hanover

array([[ 0.00000000e+00,  0.00000000e+00,  1.41397335e-15, ...,
        -3.17981676e-03, -1.35701062e-04,  6.50109141e-05],
       [ 0.00000000e+00,  0.00000000e+00,  1.41397335e-15, ...,
        -3.78494314e-03, -2.77650263e-03, -2.81038345e-03],
       [ 0.00000000e+00,  0.00000000e+00,  1.41397335e-15, ...,
        -1.57625065e-03, -4.14244039e-03,  1.17331110e-04],
       ...,
       [ 0.00000000e+00,  1.00000000e+00,  6.86975755e-01, ...,
         4.16947296e-03,  1.38832847e-05,  2.27493141e-03],
       [ 0.00000000e+00,  1.00000000e+00,  1.49102038e+00, ...,
         2.11802800e-03,  2.51115370e-03, -3.70924245e-04],
       [ 0.00000000e+00,  1.00000000e+00,  1.41397335e-15, ...,
         1.22791948e-03,  1.75915950e-03,  2.06002500e-03]],
      shape=(100, 104))

In [77]:
from notebook.services.config import ConfigManager
c = ConfigManager()
c.update('notebook', {"CodeCell": {"cm_config": {"autoCloseBrackets": False}}})
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))